In [ ]:
import duckdb
print(duckdb.connect().execute("SELECT 1").fetchall())

: 

In [1]:
import duckdb

con = duckdb.connect("tpch.duckdb")

con.execute("INSTALL tpch")
con.execute("LOAD tpch")
con.execute("CALL dbgen(sf=1)")

In [3]:
import time

def run_scan(con):
    start = time.time()
    
    res = con.execute("""
    SELECT SUM(l_extendedprice * l_discount)
    FROM lineitem
    WHERE l_discount BETWEEN 0.05 AND 0.07
      AND l_quantity < 24
    """).fetchone()
    
    end = time.time()
    return end - start, res

In [4]:
import numpy as np

def run_bitmap(con):

    df = con.execute("""
    SELECT l_extendedprice, l_discount, l_quantity
    FROM lineitem
    """).fetchdf()
    
    price = df["l_extendedprice"].to_numpy()
    discount = df["l_discount"].to_numpy()
    quantity = df["l_quantity"].to_numpy()

    start = time.time()

    bitmap_discount = (discount >= 0.05) & (discount <= 0.07)
    bitmap_quantity = (quantity < 24)

    final_bitmap = bitmap_discount & bitmap_quantity

    result = np.sum(price[final_bitmap] * discount[final_bitmap])

    end = time.time()

    return end - start, result

In [5]:
def benchmark(con, func, runs=5):
    times = []
    for _ in range(runs):
        t, _ = func(con)
        times.append(t)
    return times